In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('laptop_data.csv')
df.head()

Ten zbiór danych opisuje laptopy i ich cechy techniczne. Każdy wiersz to jeden model laptopa, a kolumny zawierają informacje,
które posłyżą do predykcji ceny (Price).

 Wyjaśnienie kolumn:
- Unnamed: 0: indeks/numer rekordu w zbiorze, liczba całkowita
- Company: producent laptopa, typ danych to kategoria
- TypeName: typ laptopa
- Inches: rozmiar ekranu w calach, liczba zmiennoprzecinkowa
- ScreenResolution: rozdzielczość i typ ekranu, tekst/kategoria
- Cpu: procesor, tekst
- Ram: ilość pamięci RAM, tekst
- Memory: typ i pojemność dysku, tekst
- Gpu: karta graficzna, tekst
- OpSys: system operacyjny, kategoria
- Weight: waga laptopa, tekst

In [ ]:
df.info()

In [ ]:
df.shape

In [ ]:
# usunięcie kolumny
df.drop(columns=['Unnamed: 0'], inplace=True)

In [ ]:
df.head()

In [ ]:
df['Ram'] = df['Ram'].str.replace('GB','').astype(int)
df['Weight'] = df['Weight'].str.replace('kg','').astype(float)

In [ ]:
df.info()

In [ ]:
import seaborn as sns
sns.displot(df['Price'], kde=True)

In [ ]:
df['Company'].value_counts()

In [ ]:
sns.barplot(x=df['Company'], y =df['Price'])
plt.xticks(rotation='vertical')
plt.show()

In [ ]:
df['TypeName'].value_counts()

In [ ]:
sns.barplot(x=df['TypeName'], y =df['Price'])
plt.xticks(rotation='vertical')
plt.show()

In [ ]:
# rozmiar ekranu
sns.displot(x=df['Inches'], kde=True)

In [ ]:
sns.scatterplot(x=df['Inches'], y=df['Price'])

analiza ScreenResolution

In [ ]:
df['ScreenResolution'].value_counts()

In [ ]:
df['Touchscreen'] = df['ScreenResolution'].apply(lambda x: 1 if 'Touchscreen' in x else 0)

In [ ]:
df.sample(5)

In [ ]:
df['Touchscreen'].value_counts()

In [ ]:
sns.barplot(x=df['Touchscreen'], y=df['Price'])

In [ ]:
df['Ips'] = df['ScreenResolution'].apply(lambda x: 1 if 'IPS' in x else 0)

In [ ]:
df.head()

analiza Ips

In [ ]:
df['Ips'].value_counts()

In [ ]:
sns.barplot(x=df['Ips'], y=df['Price'])

In [ ]:
df['X_res'] = df['ScreenResolution'].str.extract(r'(\d+)x')[0].astype(int)
df['Y_res'] = df['ScreenResolution'].str.extract(r'x(\d+)')[0].astype(int)

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.corr(numeric_only=True)['Price'].sort_values(ascending=False)

Wnioski korelacji
- Ram -> 0.743 -  To bardzo wysoka korelacja. Oznacza im więcej RAM-u, tym zwykle wyższa cena laptopa.
- X_res i Y_res: 0.556 i 0.553 - Średnio silna zależność. Czyli wyższa rozdzielczość - droższy laptop.

## PPI(Pixels Per Inch), czyli ile pikseli przypada na 1 cal ekranu.



In [ ]:
df['ppi'] = (((df['X_res']**2) + (df['Y_res']**2))**0.5 / df['Inches']).astype('float')

In [ ]:
df.head()

In [ ]:
df.corr(numeric_only=True)['Price'].sort_values(ascending=False)

In [ ]:
# usunięcie już niepotrzebnych kolumn
df.drop(columns=['ScreenResolution','Inches','X_res','Y_res'], inplace=True)

In [ ]:
df.head()

Analiza CPU

In [ ]:
df['Cpu'].value_counts()

In [ ]:
df['Cpu'].apply(lambda x: x.split())

In [ ]:
df['Cpu_Name'] = df['Cpu'].apply(lambda x: " ".join(x.split()[0:3]))

In [ ]:
df.head(10)

In [ ]:
def fetch_processor(text):
    if text == 'Intel Core i7' or text == 'Intel Core i5' or text == 'Intel Core i3':
        return text
    else:
        if text.split()[0] == 'Intel':
            return 'Other Intel Processor'
        else:
            return 'AMD Processor'

In [ ]:
df['Cpu_brand'] = df['Cpu_Name'].apply(fetch_processor)

In [ ]:
df.head(10)

In [ ]:
df['Cpu_brand'].value_counts()

In [ ]:
sns.barplot(x=df['Cpu_brand'], y=df['Price'])
plt.xticks(rotation='vertical')
plt.show()

In [ ]:
df.drop(columns=['Cpu', 'Cpu_Name'], inplace=True)

In [ ]:
df.head()

analiza Ram

In [ ]:
df['Ram'].value_counts()

In [ ]:
sns.barplot(x=df['Ram'], y=df['Price'])
plt.xticks(rotation='vertical')
plt.show()

analiza Memory

In [ ]:
df['Memory'].value_counts()

In [ ]:
# feature engineering dla kolumny Memory
df['Memory'] = df['Memory'].astype(str).replace('\\.0', '', regex=True)
df['Memory'] = df['Memory'].str.replace('GB', '', regex=True)
df['Memory'] = df['Memory'].str.replace('TB', '000', regex=True)
new = df['Memory'].str.split("+", n = 1, expand = True)

df['first'] = new[0]
df['first'] = df['first'].str.strip()

df['second'] = new[1]

df['Layer1HDD'] = df['first'].apply(lambda x: 1 if 'HDD' in x else 0)
df['Layer1SSD'] = df['first'].apply(lambda x: 1 if 'SSD' in x else 0)
df['Layer1Hybrid'] = df['first'].apply(lambda x: 1 if 'Hybrid' in x else 0)
df['Layer1Flash_Storage'] = df['first'].apply(lambda x: 1 if 'Flash Storage' in x else 0)

df['first'] = df['first'].str.replace(r'\D','', regex=True)

df['second'].fillna('0',inplace=True)

df['Layer2HDD'] = df['second'].apply(lambda x: 1 if 'HDD' in x else 0)
df['Layer2SSD'] = df['second'].apply(lambda x: 1 if 'SSD' in x else 0)
df['Layer2Hybrid'] = df['second'].apply(lambda x: 1 if 'Hybrid' in x else 0)
df['Layer2Flash_Storage'] = df['second'].apply(lambda x: 1 if 'Flash Storage' in x else 0)

df['second'] = df['second'].str.replace(r'\D','', regex=True)

df['first'] = df['first'].astype(int)
df['second'] = df['second'].astype(int)

df['HDD'] = (df['first'] * df['Layer1HDD']) + (df['second'] * df['Layer2HDD'])
df['SSD'] = (df['first'] * df['Layer1SSD']) + (df['second'] * df['Layer2SSD'])
df['Hybrid'] = (df['first'] * df['Layer1Hybrid']) + (df['second'] * df['Layer2Hybrid'])
df['Flash_Storage'] = (df['first'] * df['Layer1Flash_Storage']) + (df['second'] * df['Layer2Flash_Storage'])

df.drop(columns=['first','second','Layer1HDD','Layer1SSD','Layer1Hybrid', 'Layer1Flash_Storage','Layer2HDD','Layer2SSD','Layer2Hybrid','Layer2Flash_Storage'],inplace=True)

Wyjasnienie kodu powyżej
- Celem jest zamiana trudnego tekstu typu: '256GB SSD', '1TB HDD', '128GB SSD + 1TB HDD' na osobne liczbowe kolumny: HDD, SSD, Hybrid, Flash_storage

In [ ]:
df.head()

In [ ]:
df.drop(columns=['Memory'], inplace=True)

In [ ]:
df.head(10)

In [ ]:
df.corr(numeric_only=True)['Price'].sort_values(ascending=False)

In [ ]:
#usunięcie kolumn mniej 'znaczących'
df.drop(columns=['Hybrid','Flash_Storage'], inplace=True)

In [ ]:
df.head()

analiza Gpu

In [ ]:
df['Gpu'].value_counts()

In [ ]:
df['Gpu_brand'] = df['Gpu'].apply(lambda x: x.split()[0])

In [ ]:
df.head()

In [ ]:
df['Gpu_brand'].value_counts()

In [ ]:
df = df[df['Gpu_brand'] != 'ARM']

In [ ]:
df['Gpu_brand'].value_counts()

In [ ]:
sns.barplot(x=df['Gpu_brand'], y=df['Price'],estimator=np.median)
plt.xticks(rotation='vertical')
plt.show()

In [ ]:
df.drop(columns=['Gpu'], inplace=True)

In [ ]:
df.head()

analiza systemu operacyjnego

In [ ]:
df['OpSys'].value_counts()

In [ ]:
sns.barplot(x=df['OpSys'], y=df['Price'])
plt.xticks(rotation='vertical')
plt.show()

In [ ]:
def category_os(sys):
  if sys == 'Windows 10' or sys == 'Windows 7' or sys == 'Windows 10 S':
    return 'Windows'
  elif sys == 'macOS' or sys == 'Mac OS X':
    return 'Mac'
  else:
    return 'Others/No OS/Linux'

In [ ]:
df['os'] = df['OpSys'].apply(category_os)

In [ ]:
df.head()

In [ ]:
df.drop(columns=['OpSys'], inplace=True)

In [ ]:
df.head()

In [ ]:
sns.barplot(x=df['os'], y=df['Price'])
plt.xticks(rotation='vertical')
plt.show()

analiza weight

In [ ]:
sns.displot(df['Weight'])

In [ ]:
sns.scatterplot(x=df['Weight'], y=df['Price'])

In [ ]:
sns.heatmap(df.corr(numeric_only=True), annot=True)

In [ ]:
sns.displot(np.log(df['Price']), kde=True)

In [ ]:
X = df.drop(columns=['Price'])
y = np.log(df['Price'])

In [ ]:
X

In [ ]:
y

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2, random_state=42)

In [ ]:
X_train

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import r2_score, mean_absolute_error

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor, AdaBoostRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor

LinearRegression

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ('column_tnf', OneHotEncoder(sparse_output=False, drop='first'), [0,1,7,10,11])
], remainder='passthrough')

lr_model = LinearRegression()

pipe = Pipeline([
     ('preprocessing', preprocessor),
    ('linear_regression', lr_model)
])

pipe.fit(X_train, y_train)

y_pred = pipe.predict(X_test)

print('R2 score', r2_score(y_test, y_pred))
print('MAE', mean_absolute_error(y_test, y_pred))

In [ ]:
# Porównanie prawdziwych i przewidywanych wartości
comparison = pd.DataFrame({

    'Actual Price': np.exp(y_test),
    'Predicted Price': np.exp(y_pred)

})

comparison['Difference'] = (comparison['Actual Price']- comparison['Predicted Price'])
comparison['Error %'] = (abs(comparison['Difference'])/ comparison['Actual Price']) * 100
comparison.head(10)

In [ ]:
X_train

Ridge Regression

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ('col_tnf',OneHotEncoder(sparse_output=False,drop='first'),[0,1,7,10,11])
],remainder='passthrough')

rr_model = Ridge(alpha=10)

pipe = Pipeline([
    ('preprocessins',preprocessor),
    ('ridge_regression',rr_model)
])

pipe.fit(X_train,y_train)

y_pred = pipe.predict(X_test)

print('R2 score',r2_score(y_test,y_pred))
print('MAE',mean_absolute_error(y_test,y_pred))

Lasso Regression

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ('col_tnf',OneHotEncoder(sparse_output=False,drop='first'),[0,1,7,10,11])
],remainder='passthrough')

lasso_model = Lasso(alpha=0.001)

pipe = Pipeline([
    ('preprocessing',preprocessor),
    ('lasso_regression',lasso_model)
])

pipe.fit(X_train,y_train)

y_pred = pipe.predict(X_test)

print('R2 score',r2_score(y_test,y_pred))
print('MAE',mean_absolute_error(y_test,y_pred))

KNN

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ('col_tnf',OneHotEncoder(sparse_output=False,drop='first'),[0,1,7,10,11])
],remainder='passthrough')

knn_model = KNeighborsRegressor(n_neighbors=3)

pipe = Pipeline([
    ('spreprocessing',preprocessor),
    ('KNN',knn_model)
])

pipe.fit(X_train,y_train)

y_pred = pipe.predict(X_test)

print('R2 score',r2_score(y_test,y_pred))
print('MAE',mean_absolute_error(y_test,y_pred))

Decision Tree

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ('col_tnf',OneHotEncoder(sparse_output=False,drop='first'),[0,1,7,10,11])
],remainder='passthrough')

dt_model = DecisionTreeRegressor(max_depth=8)

pipe = Pipeline([
    ('spreprocessing',preprocessor),
    ('decision_tree',dt_model)
])

pipe.fit(X_train,y_train)

y_pred = pipe.predict(X_test)

print('R2 score',r2_score(y_test,y_pred))
print('MAE',mean_absolute_error(y_test,y_pred))

SVM

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ('col_tnf',OneHotEncoder(sparse_output=False,drop='first'),[0,1,7,10,11])
],remainder='passthrough')

svm_model = SVR(kernel='rbf',C=10000,epsilon=0.1)

pipe = Pipeline([
    ('preprocessing',preprocessor),
    ('svm',svm_model)
])

pipe.fit(X_train,y_train)

y_pred = pipe.predict(X_test)

print('R2 score',r2_score(y_test,y_pred))
print('MAE',mean_absolute_error(y_test,y_pred))

Random Forest

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ('col_tnf',OneHotEncoder(sparse_output=False,drop='first'),[0,1,7,10,11])
],remainder='passthrough')

rf_model = RandomForestRegressor(n_estimators=100,
                              random_state=3,
                              max_samples=0.5,
                              max_features=0.75,
                              max_depth=15)

pipe = Pipeline([
    ('preprocessing',preprocessor),
    ('random_forest',rf_model)
])

pipe.fit(X_train,y_train)

y_pred = pipe.predict(X_test)

print('R2 score',r2_score(y_test,y_pred))
print('MAE',mean_absolute_error(y_test,y_pred))

ExtraTrees

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ('col_tnf',OneHotEncoder(sparse_output=False,drop='first'),[0,1,7,10,11])
],remainder='passthrough')

etr_model = ExtraTreesRegressor(n_estimators=100,
                              random_state=3,
                              max_samples=0.5,
                              max_features=0.75,
                              max_depth=15,
                              bootstrap=True)

pipe = Pipeline([
    ('preprocessing',preprocessor),
    ('extraTrees',etr_model)
])

pipe.fit(X_train,y_train)

y_pred = pipe.predict(X_test)

print('R2 score',r2_score(y_test,y_pred))
print('MAE',mean_absolute_error(y_test,y_pred))

AdaBoost

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ('col_tnf',OneHotEncoder(sparse_output=False,drop='first'),[0,1,7,10,11])
],remainder='passthrough')

ab_model = AdaBoostRegressor(n_estimators=15,learning_rate=1.0)

pipe = Pipeline([
    ('preprocessing',preprocessor),
    ('AdaBoost',ab_model)
])

pipe.fit(X_train,y_train)

y_pred = pipe.predict(X_test)

print('R2 score',r2_score(y_test,y_pred))
print('MAE',mean_absolute_error(y_test,y_pred))

Gradient Boost

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ('col_tnf',OneHotEncoder(sparse_output=False,drop='first'),[0,1,7,10,11])
],remainder='passthrough')

gb_model = GradientBoostingRegressor(n_estimators=500)

pipe = Pipeline([
    ('preprocessing',preprocessor),
    ('Gradient_boost',gb_model)
])

pipe.fit(X_train,y_train)

y_pred = pipe.predict(X_test)

print('R2 score',r2_score(y_test,y_pred))
print('MAE',mean_absolute_error(y_test,y_pred))

XgBoost

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ('col_tnf',OneHotEncoder(sparse_output=False,drop='first'),[0,1,7,10,11])
],remainder='passthrough')

xgb_model = XGBRegressor(n_estimators=45,max_depth=5,learning_rate=0.5)

pipe = Pipeline([
    ('preprocessing',preprocessor),
    ('XgBoost',xgb_model)
])

pipe.fit(X_train,y_train)

y_pred = pipe.predict(X_test)

print('R2 score',r2_score(y_test,y_pred))
print('MAE',mean_absolute_error(y_test,y_pred))

In [ ]:
comparison = pd.DataFrame({

    'Actual Price': np.exp(y_test),
    'Predicted Price': np.exp(y_pred)

})
comparison['Difference'] = (
    comparison['Actual Price'] - comparison['Predicted Price']
)

comparison['Error %'] = (
    abs(comparison['Difference']) / comparison['Actual Price']) * 100
comparison.head(10)

In [ ]:
# przykład predykcji nowego laptopa
new_laptop = pd.DataFrame({
    'Company': ['Dell'],
    'TypeName': ['Gaming'],
    'Ram': [16],
    'Weight': [2.3],
    'Touchscreen': [0],
    'Ips': [1],
    'ppi': [141],
    'Cpu_brand': ['Intel Core i7'],
    'HDD': [0],
    'SSD': [512],
    'Gpu_brand': ['Nvidia'],
    'os': ['Windows']

})

predicted_price = np.exp(pipe.predict(new_laptop)[0])
print("Przewidywana cena:", predicted_price)

Exporting the Model

In [ ]:
import pickle

pickle.dump(df,open('df.pkl','wb'))
pickle.dump(pipe,open('pipe.pkl','wb'))

Porównanie

In [ ]:
models = {

    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(alpha=10),
    'Lasso': Lasso(alpha=0.001),
    'KNN': KNeighborsRegressor(n_neighbors=3),
    'Decision Tree': DecisionTreeRegressor(max_depth=8),
    'SVR': SVR(kernel='rbf', C=10000, epsilon=0.1),
    'Random Forest': RandomForestRegressor(
        n_estimators=100,
        random_state=3,
        max_samples=0.5,
        max_features=0.75,
        max_depth=15
    ),
    'Extra Trees': ExtraTreesRegressor(
        n_estimators=100,
        random_state=3,
        max_samples=0.5,
        max_features=0.75,
        max_depth=15,
        bootstrap=True
    ),
    'AdaBoost': AdaBoostRegressor(
        n_estimators=15,
        learning_rate=1.0
    ),
    'Gradient Boosting': GradientBoostingRegressor(
        n_estimators=500
    ),
    'XGBoost': XGBRegressor(
        n_estimators=45,
        max_depth=5,
        learning_rate=0.5
    )
}

In [ ]:
preprocessor = ColumnTransformer(transformers=[('cat', OneHotEncoder(sparse_output=False, drop='first'), [0,1,7,10,11])], remainder='passthrough')

In [ ]:
results = []

for name, model in models.items():
    pipe = Pipeline([
        ('preprocessing', preprocessor),
        ('model', model)
    ])

    pipe.fit(X_train, y_train)

    y_pred = pipe.predict(X_test)

    r2 = r2_score(y_test, y_pred)

    mae = mean_absolute_error(y_test, y_pred)

    results.append([name, r2, mae])

In [ ]:
results_df = pd.DataFrame(
    results,
    columns=['Model', 'R2 Score', 'MAE']
)
results_df.sort_values(by='R2 Score', ascending=False)

## Podsumowanie
-  XGBoost ma najwyższe R^2 = 0.8816, MAE = 0.1622. To znaczy, że model wyjaśnia około 88% zmienności cen laptopów. Oznacza, że model dobrze nauczył się zależności między: RAM, SSD, PPI, GPU brand, CPU, typem laptopa, systemem operacyjnym
a ceną. XGBoost wygrał, bo świetnie radzi sobie z nieliniowościami np. wzrost ceny nie jest liniowy: 8 -> 16 GB RAM = duży skok, 16 -> 32 GB = jeszcze większy
- Gradient Boosting prawie wygrał, czyli klasyczny boosting też działa świetnie.
- Random Forest i Extra Trees, bardzo solidne, oba odporne, mocne.

In [ ]:
new_laptop = pd.DataFrame({

    'Company': ['Dell'],
    'TypeName': ['Gaming'],
    'Ram': [16],
    'Weight': [2.3],
    'Touchscreen': [0],
    'Ips': [1],
    'ppi': [141],
    'Cpu_brand': ['Intel Core i7'],
    'HDD': [0],
    'SSD': [512],
    'Gpu_brand': ['Nvidia'],
    'os': ['Windows']

})

In [ ]:
# porównianie nowego produktu dla wszystkich modeli

predictions_price = []

for name, model in models.items():

    pipe = Pipeline([
        ('preprocessing', preprocessor),
        ('model', model)
    ])


    pipe.fit(X_train, y_train)

    pred = np.exp(pipe.predict(new_laptop)[0])

    predictions_price.append([name, pred])

In [ ]:
predictions_price_df = pd.DataFrame(predictions_price,columns=['Model', 'Predicted Price'])
predictions_price_df.sort_values(by='Predicted Price',ascending=False)

Zakres predykcji dla najlepszych modeli

In [ ]:
top_models = {
    'XGBoost': XGBRegressor(
        n_estimators=45,
        max_depth=5,
        learning_rate=0.5
    ),
    'Gradient Boosting': GradientBoostingRegressor(
        n_estimators=500
    ),
    'Random Forest': RandomForestRegressor(
        n_estimators=100,
        random_state=3,
        max_samples=0.5,
        max_features=0.75,
        max_depth=15
    ),
    'Extra Trees': ExtraTreesRegressor(
        n_estimators=100,
        random_state=3,
        max_samples=0.5,
        max_features=0.75,
        max_depth=15,
        bootstrap=True
    )
}

In [ ]:
prices = []

for name, model in top_models.items():

    pipe = Pipeline([
        ('preprocessing', preprocessor),
        ('model', model)
    ])

    pipe.fit(X_train, y_train)

    pred = np.exp(pipe.predict(new_laptop)[0])

    min_range = pred * 0.9
    max_range = pred * 1.1

    prices.append([
        name,
        pred,
        min_range,
        max_range
    ])


predictions_df = pd.DataFrame(

    prices,

    columns=[

        'Model',
        'Predicted Price',
        'Min Range',
        'Max Range'
    ])

predictions_df.sort_values(by='Predicted Price', ascending=False)

Podsumowanie wyników predykcji
- Najlepsze modele przewidziały cenę nowego laptopa w stosunkowo podobnym zakresie od około 84k do 97k.
- Modele oparte na drzewach decyzyjnych wykazały dość dużą spójność predykcji, co sugeruje stabilną estymację wartości laptopa.
Najbardziej konserwatywne predykcje wygenerowały Random Forest i Extra Trees, natomiast najwyższą wycenę przewidział Gradient Boosting.
Na podstawie wszystkich modeli można przyjąć, że realistyczna cena badanego laptopa znajduje się w przedziale: 85 000 - 97 000.
- Uwzględnienie zakresu predykcji pozwala lepiej interpretować niepewność modeli machine learning i bardziej realistycznie szacować wartość rynkową produktu.